# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishnu-varthan1/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [12]:
!git clone https://github.com/vishnu-varthan1/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08: imports and dataset setup

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42

REPO = Path("/content/flyrank-ml-internship")
DATA_PATH = REPO / "data/raw/content_refresh_anonymized.csv"

assert DATA_PATH.exists(), "Dataset not found. Check the repository clone."

df = pd.read_csv(DATA_PATH)

# Create the target from the documented label source.
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())
print("\nTarget distribution:")
display(df["is_declining_label"].value_counts())
print("\nTarget base rate:", round(df["is_declining_label"].mean(), 4))

Dataset shape: (30000, 45)
Unique clients: 32

Target distribution:


,count
is_declining_label,
1,16262
0,13738



Target base rate: 0.5421


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Use only historical 30-day measurements as predictive features.
# Do not use current-period or label-derived trend columns.

FEATURES = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

TARGET = "is_declining_label"
GROUP = "client_id"

# Verify all required columns exist.
required = FEATURES + [TARGET, GROUP]
missing = [c for c in required if c not in df.columns]

assert not missing, f"Missing columns: {missing}"

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df[GROUP].copy()

# Log-transform heavy-tailed traffic counts.
X = np.log1p(X.clip(lower=0))

# Hold out entire clients, not random rows.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(train_idx))
print("Testing rows:", len(test_idx))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(train_clients.intersection(test_clients)))

assert train_clients.isdisjoint(test_clients)
assert y_train.nunique() == 2
assert y_test.nunique() == 2

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Train Logistic Regression

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Model evaluation")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, y_pred, zero_division=0), 4))

if y_test.nunique() == 2:
    print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

Model evaluation
Accuracy: 0.5564
Precision: 0.5627
Recall: 0.5916
F1: 0.5768
ROC-AUC: 0.6139

Confusion matrix:
[[1566 1448]
 [1286 1863]]

Classification report:
              precision    recall  f1-score   support

           0       0.55      0.52      0.53      3014
           1       0.56      0.59      0.58      3149

    accuracy                           0.56      6163
   macro avg       0.56      0.56      0.56      6163
weighted avg       0.56      0.56      0.56      6163



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect model errors

error_analysis = df.iloc[test_idx][
    ["content_id", "client_id"]
].copy()

error_analysis["actual"] = y_test.to_numpy()
error_analysis["predicted"] = y_pred
error_analysis["decline_probability"] = y_prob

error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual"] == 1) &
        (error_analysis["predicted"] == 0),

        (error_analysis["actual"] == 0) &
        (error_analysis["predicted"] == 1)
    ],
    [
        "False Negative",
        "False Positive"
    ],
    default="Correct"
)

print("Error counts:")
display(error_analysis["error_type"].value_counts())

print("\nThree incorrectly classified examples:")
display(
    error_analysis[
        error_analysis["error_type"] != "Correct"
    ].head(3)
)

print("\nModel coefficients:")
coefficients = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": model.named_steps["classifier"].coef_[0]
}).sort_values("coefficient", ascending=False)

display(coefficients)

Error counts:


,count
error_type,
Correct,3429
False Positive,1448
False Negative,1286



Three incorrectly classified examples:


,content_id,client_id,actual,predicted,decline_probability,error_type
0,content_304f48230142,client_f369cb89fc,1,0,0.437426,False Negative
13,content_a5a2fbc76336,client_8527a891e2,0,1,0.559790,False Positive
23,content_2da6ae9d0882,client_e629fa6598,1,0,0.422014,False Negative



Model coefficients:


,feature,coefficient
0,impressions_prev_30d,1.210009
2,sessions_prev_30d,-0.374429
1,clicks_prev_30d,-0.490486


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.